## Mirroring & pull-through caching

At scale, a team pulls `nginx:alpine` thousands of times a day across CI and prod. Hitting Docker Hub for every pull is **slow** (latency), **fragile** (a Hub outage fails your deploys), and **rate-limited** (the anonymous-pull cap from section 3). The fix is a **pull-through cache**: a local registry that, on a miss, fetches from upstream once and caches the result.

It's the same `registry:2` image, put in proxy mode:

```bash
docker run -d -p 5000:5000 --name cache \
  -e REGISTRY_PROXY_REMOTEURL=https://registry-1.docker.io \
  -v cache-data:/var/lib/registry registry:2
```

Then point the daemon at it as a **registry mirror** in `/etc/docker/daemon.json`:

```json
{ "registry-mirrors": ["http://my-cache:5000"] }
```

Now the first pull of `nginx:alpine` traverses the cache and stores its blobs; every later pull across the fleet is served from cache at **LAN speed**, and Hub is hit once instead of thousands of times. GHCR, ECR, and GAR all offer similar pull-through modes.

**Mirror vs. proxy** are the same idea under two names: the cache holds the *same* references as upstream (`nginx:alpine` resolves identically), the daemon tries the cache first and falls back to the origin. The distinction that matters in practice is **pull-through cache** (transparent, on-demand — what you almost always want) vs. a **full mirror** (a complete copy you actively replicate). For organisations doing thousands of pulls a day, a pull-through cache is one small container that saves real money, latency, and a dependency on someone else's uptime — a natural companion to the private registry from the last section.